# Swept Pazy wing flutter analysis

Evaluate the aeroelastic eigenvalues (and resulting frequency and damping) for key modes for three variants of the Pazy wing to compare the effect of sweep:

1. Swept Pazy wing with the tip mass placed at the leading edge.
2. Swept Pazy wing with the tip mass placed at the trailing edge.
3. Straight Pazy wing rotated by 10 degrees (kinematic-only sweep — retains the straight-wing structural properties).

Imports

In [ ]:
from copy import deepcopy
from functools import partial
from pathlib import Path
from typing import Literal, cast

import jax
import numpy as np
from jax import Array
from jax import numpy as jnp
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize

from flapjax.aero.flowfields import ConstantFlowField
from flapjax.coupled import CoupledAeroelastic
from flapjax.models.pazy.straight.pazy_wing import generate_pazy_wing
from flapjax.models.pazy.swept.swept_pazy_wing import generate_swept_pazy_wing
from flapjax.utils.print_utils import set_verbosity

Discretisation and Parameter Setup

In [ ]:
set_verbosity("silent")  # remove flapjax console prints due to the large number of cases

n_vel = 30  # number of velocity points
m = 12  # wing chordwise discretisation
m_star = 40  # number of wake panels per strip
alpha = jnp.deg2rad(7.0)  # angle of attack shared across all sweep cases

cases: tuple[Literal["swept_LE", "swept_TE", "straight_rotated"], ...] = (
    "swept_LE",
    "swept_TE",
    "straight_rotated",
)
u_inf_mags = jnp.linspace(20.0, 100.0, n_vel)  # velocity sweep

# plot colourmap
cmap = plt.get_cmap("viridis")

# noinspection PyArgumentList
norm = cast(Normalize, Normalize(vmin=float(u_inf_mags.min()), vmax=float(u_inf_mags.max())))

# optionally write the frequency/damping ratio of the first 3 modes per case to a .txt file
write_txt_output = True
output_dir = Path("./flutter_outputs/")

Factory for the base wing for a given sweep case. Shared across all velocities within a case.

In [ ]:
def make_wing(case: Literal["swept_LE", "swept_TE", "straight_rotated"]) -> CoupledAeroelastic:
    flowfield = ConstantFlowField(
        u_inf=jnp.array([0.0, 0.0, 0.0]),  # overwritten per velocity in eigs_at_velocity
        rho=1.225,
        relative_motion=True,
    )
    match case:
        case "swept_LE":
            return generate_swept_pazy_wing(
                flowfield=flowfield,
                aoa=alpha,
                tip_mass="LE_CORRECTED",
                m=m,
                m_star=m_star,
                node_multiplier=2,
                sweep_angle=10,
            )
        case "swept_TE":
            return generate_swept_pazy_wing(
                flowfield=flowfield,
                aoa=alpha,
                tip_mass="TE",
                m=m,
                m_star=m_star,
                node_multiplier=2,
                sweep_angle=10,
            )
        case "straight_rotated":
            return generate_pazy_wing(
                flowfield=flowfield,
                aoa=alpha,
                m=m,
                m_star=m_star,
                node_multiplier=2,
                skin=True,
                variable_disc_wake=True,
                sweep=jnp.deg2rad(10.0),
            )
        case _:
            raise ValueError(f"Unknown sweep case: {case}")

Function to evaluate the eigenvalues for a given wing and velocity. This is mapped across the input velocities, allowing multiple design points to be evaluated in parallel.

In [ ]:
def eigs_at_velocity(u_mag: Array, wing: CoupledAeroelastic) -> Array:
    # this function is mapped across the input velocities, allowing multiple design points to be evaluated in
    # parallel
    inner_wing = deepcopy(wing)

    # set the required design variables for this case
    dt = 0.1 / (m * u_mag)
    inner_wing.set_design_variables(
        coords=inner_wing.structure.x0_reference,
        k_cs=inner_wing.structure.k_cs,
        m_cs=inner_wing.structure.m_cs,
        m_lumped=None,
        dt=dt,
        flowfield=ConstantFlowField(
            u_inf=jnp.array([u_mag, 0.0, 0.0]),
            rho=1.225,
            relative_motion=True,
        ),
        zeta_b0=inner_wing.aero.zeta_b0,
        remove_checks=True,
    )

    # obtain static equilibrium
    static_sol = inner_wing.static_solve(
        prescribed_dofs=tuple(range(6)),
        horseshoe=False,
    )
    # linearise solution about static equilibrium
    linear_sol = inner_wing.linearise(
        reference=static_sol, skip_checks=True, batch_size=4, n_struct_modes=50
    )

    # obtain eigenvalies of linearised system
    eigvals, _ = linear_sol.modal(
        n_modes=5,
        plot_eigvals=False,
    )
    return eigvals

Plot eigenvalues for a given sweep case, with colour representing velocity.

In [ ]:
def plot_eigenvalues(evals: Array, case: str) -> None:
    fig, ax = plt.subplots()
    for i in range(len(u_inf_mags)):
        ax.scatter(
            evals[i].real / (2.0 * jnp.pi),
            evals[i].imag / (2.0 * jnp.pi),
            color=cmap(norm(float(u_inf_mags[i]))),
            s=10,
        )
    ax.set_xlim(-6.0, 2.0)
    ax.set_ylim(0.0, 50.0)
    ax.axvline(0.0, color="k", linewidth=0.5)
    ax.set_xlabel("Re(eig) / (2 * pi) [1/s]")
    ax.set_ylabel("Im(eig) / (2 * pi) [1/s]")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    fig.colorbar(sm, ax=ax, label="u_inf [m/s]")
    ax.set_title(f"Pazy wing eigenvalues vs velocity, case={case}")
    plt.show()

Plot damping ratio and damped frequency for a given sweep case, with colour representing velocity.

In [ ]:
def plot_damp_freq(
        zeta: Array,
        freq_d: Array,
        case: str,
) -> None:
    fig, (ax_z, ax_f) = plt.subplots(1, 2, figsize=(11, 4.5))
    for i in range(len(u_inf_mags)):
        c = cmap(norm(float(u_inf_mags[i])))
        u = float(u_inf_mags[i])
        ax_z.scatter(jnp.full_like(zeta[i], u), -zeta[i], color=c, s=10)
        ax_f.scatter(jnp.full_like(freq_d[i], u), freq_d[i], color=c, s=10)

    ax_z.axhline(0.0, color="k", linewidth=0.5)
    ax_z.set_xlabel("u_inf [m/s]")
    ax_z.set_ylabel("-Damping ratio")
    ax_z.set_xlim(u_inf_mags.min(), u_inf_mags.max())
    ax_z.set_ylim(-0.3, 0.3)
    ax_z.set_title(f"Damping ratio, case={case}")

    ax_f.set_xlabel("u_inf [m/s]")
    ax_f.set_ylabel("Damped frequency f_d [Hz]")
    ax_f.set_xlim(u_inf_mags.min(), u_inf_mags.max())
    ax_f.set_ylim(0.0, 50.0)
    ax_f.set_title(f"Damped frequency, case={case}")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    fig.colorbar(sm, ax=(ax_z, ax_f), label="u_inf [m/s]")
    plt.show()

Optionally write the frequency and damping ratio of the first 3 modes to a `.txt` file for a given sweep case.

In [ ]:
def save_modal_txt(
        u_inf: Array,
        freq_d: Array,
        zeta: Array,
        case: str,
        out_dir: Path,
        n_modes: int = 3,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    header = ["u_inf"] + [f"{name}_{i + 1}" for i in range(n_modes) for name in ("freq", "damp")]
    columns = [np.asarray(u_inf)] + [
        np.asarray(col) for i in range(n_modes) for col in (freq_d[:, i], zeta[:, i])
    ]
    data = np.column_stack(columns)
    np.savetxt(out_dir / f"damp_freq_{case}_aoa_{float(jnp.rad2deg(alpha))}.txt", data, header=" ".join(header))

In [ ]:
results_: dict[str, dict[str, Array]] = {}
for case_ in cases:  # sweep configuration comparison
    wing_ = make_wing(case=case_)

    # evaluate eigenvalues
    evals_: Array = jax.lax.map(partial(eigs_at_velocity, wing=wing_), u_inf_mags, batch_size=2)  # [n_vel, n_modes]

    zeta_ = -evals_.real / jnp.abs(evals_)  # damping ratio [n_vel, n_modes]
    freq_d_ = evals_.imag / (2.0 * jnp.pi)  # damped frequency (Hz), [n_vel, n_modes]

    results_[case_] = {"evals": evals_, "zeta": zeta_, "freq_d": freq_d_}

    # plot this case immediately rather than waiting for the full sweep to finish
    plot_eigenvalues(evals=evals_, case=case_)
    plot_damp_freq(zeta=zeta_, freq_d=freq_d_, case=case_)

    # optionally save frequency/damping ratio of the first 3 modes to a .txt file
    if write_txt_output:
        save_modal_txt(u_inf=u_inf_mags, freq_d=freq_d_, zeta=zeta_, case=case_, out_dir=output_dir)